In [1]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [11]:
pip list

Package                   Version
------------------------- -----------
anaconda-anon-usage       0.8.1
anaconda-auth             0.15.1
anaconda-channel-guide    0.1.1
anaconda-cli-base         0.9.1
annotated-doc             0.0.4
annotated-types           0.7.0
anyio                     4.12.1
archspec                  0.2.6
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asttokens                 3.0.2
async-lru                 2.3.0
attrs                     26.1.0
babel                     2.18.0
beautifulsoup4            4.15.0
bleach                    6.4.0
boltons                   25.0.0
brotlicffi                1.2.0.0
build                     1.5.1
certifi                   2026.6.17
cffi                      2.1.0
charset-normalizer        3.4.7
click                     8.4.1
colorama                  0.4.6
comm                      0.2.3
conda                     26.5.3
conda-anaconda-telemetry  0.3.0
conda-anaconda-to

In [22]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import random

BASE_URL="https://books.toscrape.com/"
print("Librerias cargadas correctamente")

Librerias cargadas correctamente


In [23]:
response = requests.get(BASE_URL,timeout=10)
print("Codigo de estado:",response.status_code)
print("Longitud del HTML:", len(response.text))

print(response.text[:500])

Codigo de estado: 200
Longitud del HTML: 51294
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


In [27]:
soup=BeautifulSoup(response.text,"html.parser")

libros_html=soup.select("article.product_pod")
print(f"Se encontraron {len(libros_html)} libros en esta pagina")

print(libros_html[0].prettify()[:600])

Se encontraron 20 libros en esta pagina
<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_pric


In [35]:
def extraer_rating(clases_css):
    """El rating viene como una clase CSS:'star-rating Three' -> 3"""
    mapa={"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    for clase in clases_css:
        if clase in mapa:
            return mapa[clase]
            return None


libros=[]
for item in libros_html:
    titulo= item.h3.a["title"]
    precio_texto = item.select_one("p.price_color").text.strip()
    precio= float(precio_texto.replace("£","").replace("Â",""))
    disponibilidad= item.select_one("p.instock.availability").text.strip()
    rating_clases= item.select_one("p.star-rating")["class"]
    rating=extraer_rating(rating_clases)
    enlace_relativo=item.h3.a["href"]

    libros.append({
    "titulo": titulo,
    "precio_gbp": precio,
    "disponibilidad": disponibilidad,
    "rating": rating,
    "enlace": BASE_URL+enlace_relativo
    })


df_pagina1=pd.DataFrame(libros)
df_pagina1.head(10)
    

,titulo,precio_gbp,disponibilidad,rating,enlace
0,A Light in the Attic,51.77,In stock,3,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,In stock,1,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,In stock,1,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,In stock,4,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,In stock,5,https://books.toscrape.com/catalogue/sapiens-a...
5,The Requiem Red,22.65,In stock,1,https://books.toscrape.com/catalogue/the-requi...
6,The Dirty Little Secrets of Getting Your Dream...,33.34,In stock,4,https://books.toscrape.com/catalogue/the-dirty...
7,The Coming Woman: A Novel Based on the Life of...,17.93,In stock,3,https://books.toscrape.com/catalogue/the-comin...
8,The Boys in the Boat: Nine Americans and Their...,22.60,In stock,4,https://books.toscrape.com/catalogue/the-boys-...
9,The Black Maria,52.15,In stock,1,https://books.toscrape.com/catalogue/the-black...


In [36]:
df_pagina1.to_csv("libros_toscrape.csv",index=False, encoding="utf-8-sig")
df_pagina1.to_excel("Libros_toscrape_pag1.xlsx", index=False)

print("Archivos guardados: libros_toscrape.csv y libros_toscrape.xlsx")

Archivos guardados: libros_toscrape.csv y libros_toscrape.xlsx
